# Avalanche Canada Wildfire Visualisations

This notebook takes places after overlaying the fire polygons from National Burn Area Composite (NBAC) with Avalanche Canada subregions. 

Prior to discovering severe burn patches within NBAC fire polygons, this notebook explores all fires present within selected NBAC data years with Avalanche Canada subregions.

## Imports & Directories

In [ ]:
#-- Packages --#

#--- Operational ---#
import os
import sys 
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
import re
import geopandas as gpd

#--- Visualisations ---#
import plotly.express as px
import plotly.graph_objects as go

#-- Directories --#
nb_dir = Path.cwd()
REPO_ROOT = nb_dir.parent
data_dir = REPO_ROOT / 'data/'
docs_dir = REPO_ROOT / 'docs/'
processed_dir = data_dir / 'processed/'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
#-- Helper Functions --#
def extract_max_year(path):
    """
    Identify shapefile with latest year of available data
    """
    years = re.findall(r"\d{4}", path.stem)
    return max(map(int, years)) if years else -1

In [ ]:
#-- Files --#

# Avalanche Canada polygons (GeoJSON)
print(f'Loading Avalanche Canada (AvCan) regions shapefile...')
avcan_path = REPO_ROOT / "data/processed/avalanche_canada_fires/AvCan_cleaned_subregions.geojson"
avcan_regions = gpd.read_file(avcan_path)
print(f" Avalanche Canada Regions loaded. {avcan_regions.crs}\n")

# Avalanche Canada fires 
fires_dir = REPO_ROOT / "data/processed/avalanche_canada_fires/shapefiles"
shp_files = list(fires_dir.glob("*.shp"))

if not shp_files:
    raise FileNotFoundError(f"No shapefiles found in {fires_dir}\n")

fires_path = max(shp_files, key=extract_max_year)

print(f"Loading AvCan fires shapefile... \n File name: {fires_path.name}")
fire_stats = gpd.read_file(fires_path)
print(f" Avalanche Canada Fires loaded. {fire_stats.crs}\n")


In [ ]:
fire_stats.loc[
    (fire_stats['subregion'] == 'South_Chilcotin') &
    (fire_stats['year'] <= 1992)
].count()

In [ ]:
mask = (fire_stats['subregion'] == 'Sky_Pilot') & (fire_stats['year'] <= 1992)

fire_stats[mask]     # non-null counts per column
# or if you just want number of rows:
        # or fire_stats[mask].shape[0]


In [ ]:
print(f'Aggregate fire statistics for each AvCanada subregion')
# fires_regions has: gid, year, adj_ha, reference_region, polygon_name, area_ha, geometry, ...

region_stats_raw = (
    fire_stats
    .groupby(["prov_terr","region", "subregion"])
    .agg(
        n_fires=("gid", "nunique"),     # GID = year + fire id, unique per fire event
        subreg_ha=("subreg_ha", "sum")    # area_ha = clipped fire area in that region
    )
    .reset_index()
)


In [ ]:
fire_stats['region'].unique()

In [ ]:
# 2) One row per AvCan polygon, with geometry + admin_area
region_stats = fire_stats[["region", "subregion", "prov_terr", "geometry"]].drop_duplicates()

In [ ]:
# Merge all regions with fire statistics
region_stats = region_stats.merge(
    region_stats_raw,
    on=["prov_terr","region", "subregion"],
    how="left"        # keep ALL regions, even if no match in fire_stats
)
# Replace NaN with 0 for regions without fires
region_stats[["n_fires", "subreg_ha"]] = region_stats[["n_fires", "subreg_ha"]].fillna(0)


In [ ]:
# 1. Make a WGS84 (lat/lon) copy for mapping
avcan_regions_ll = avcan_regions.to_crs(epsg=4326)   # <--- key line

print("AvCan regions CRS for mapping:", avcan_regions_ll.crs)

# 2. Convert that to GeoJSON
avcan_geojson = json.loads(avcan_regions_ll.to_json())

# 3. (Optional sanity check) make sure subregion names match
missing_in_geo = set(region_stats["subregion"]) - set(avcan_regions_ll["subregion"])
print("Subregions in stats but not in GeoJSON:", missing_in_geo)

In [ ]:



year_min = fire_stats["year"].min()
year_max = fire_stats["year"].max()

title_text = (
    "Number of Fires in Avalanche Canada Regions"
    f"<br><span style='font-size:12px; color:gray;'>Between years: {year_min}–{year_max}</span>"
)

# 2. Build choropleth
fig = px.choropleth_map(
    data_frame=region_stats,
    geojson=avcan_geojson,
    locations="subregion",                 # column in region_stats
    featureidkey="properties.subregion",   # matching property in GeoJSON
    color="n_fires",
    color_continuous_scale="YlOrRd",
    range_color=(0, region_stats["n_fires"].max()),
    map_style="outdoors",           # or "open-street-map" if no token
    zoom=4,
    center={"lat": 54, "lon": -123},         # rough center on BC
    opacity=0.7,
        labels={
        "n_fires": "Number of fires",
        "prov_terr":"Province/Territory",
        "subregion": "Subregion name",
        "region":"Region name",
        "subreg_ha":"Collective fire area (ha)"

    },
    hover_name="subregion",                       # big bold line
    hover_data={
        "region": True,                          # "Region name"
        "prov_terr": True,                             # "Province/Territory"
        "n_fires": True,                               # "Number of fires"
        "subreg_ha": ':.0f',     # "Collective fire area (ha)
    },
    title=title_text
)

# Update layout
fig.update_layout(
    margin={"r":0, "t":50, "l":0, "b":0},
    title_x=0.5,  # center the title,
    autosize=True
)

# Export visualisation
output_file = docs_dir / "Fires_in_Avalanche_Canada_Subregions.html"

fig.write_html(
    output_file,
    include_plotlyjs="cdn",  # load plotly.js from CDN (smaller file)
    full_html=True           # full standalone HTML page
)

print(f"Saved interactive map to: {output_file}")

# Show figure
fig.show()


In [ ]:
# fires_regions has: YEAR, reference_region, polygon_name, GID (or NFIREID), area_ha, ...

fires_by_year_region_raw = (
    fire_stats
    .groupby(["year", "prov_terr","region", "subregion"])
    .agg(
        n_fires=("gid", "nunique"),     # number of distinct fires hitting that region in that year
        subreg_ha_sum=("subreg_ha", "sum")   # optional, total burned area in that region-year
    )
    .reset_index()
)


In [ ]:

# All years you care about 
years = sorted(fire_stats["year"].unique())

years_df = pd.DataFrame({"year": years})

# All AvCan regions (use the same `regions` you used before)
region_keys = fire_stats[["prov_terr","region", "subregion"]].drop_duplicates()

# Cartesian product (cross join) of years × regions
full_grid = (
    years_df.assign(_key=1)
    .merge(region_keys.assign(_key=1), on="_key")
    .drop(columns="_key")
)


In [ ]:
fires_by_year_region = (
    full_grid
    .merge(
        fires_by_year_region_raw,
        on=["year", "prov_terr","region", "subregion"],
        how="left"
    )
)

# Replace NaNs (no fires) with 0
fires_by_year_region["n_fires"] = fires_by_year_region["n_fires"].fillna(0).astype(int)
fires_by_year_region["subreg_ha_sum"] = fires_by_year_region["subreg_ha_sum"].fillna(0.0)


In [ ]:
title_text = (
    "Number of Fires in Avalanche Canada Regions Per Year"
)

fig1 = px.choropleth_map(
    data_frame=fires_by_year_region,
    geojson=avcan_geojson,
    locations="subregion",                 # column in fires_by_year_region
    featureidkey="properties.subregion",   # matching field in GeoJSON
    color="n_fires",
    animation_frame="year",                   # <-- slider by year
    color_continuous_scale="YlOrRd",
    range_color=(0, fires_by_year_region["n_fires"].max()),
    map_style="outdoors",
    center={"lat": 54, "lon": -123},
    zoom=4,
    opacity=0.7,
        labels={
        "region":"Region name",
        "subregion": "Subregion name",
        "prov_terr": "Province/Territory",
        "n_fires": "Number of fires",
        "subreg_ha_sum":"Collective fire area (ha)"
    },
    hover_name="subregion",                       # big bold line
    hover_data={
        "subregion": True,      # will use label "Province"
        "n_fires": True,         # will use label "Number of fires"
        "subreg_ha_sum": ':.0f',     # optional formatting example
    },
    title=title_text
)

# Update layout
fig1.update_layout(
    margin={"r":0, "t":40, "l":0, "b":0},
    title_x=0.5,  # center the title,
    autosize=True)


# Export visualisation
output_file = docs_dir / "Fires_in_Avalanche_Canada_Subregions_Per_Year.html"

fig1.write_html(
    output_file,
    include_plotlyjs="cdn",  # load plotly.js from CDN (smaller file)
    full_html=True           # full standalone HTML page
)

print(f"Saved interactive map to: {output_file}")


fig1.show()


In [ ]:
fires_by_year_region[fires_by_year_region['region'].isin(['South_Coast_Inland','South_Coast'])]['subregion'].unique()

In [ ]:
fires_by_year_region['region'].unique()

In [ ]:
fire_stats.crs

In [ ]:

print(f"Current Fire statistics CRS: {fire_stats.crs}")   # EPSG:3978

print('Produce centroids in projected CRS for visualistion')
# 1. Make a copy for centroids while keeping polygons intact in fire_stats
centroids = fire_stats.copy()

# 2. Compute centroids in projected CRS (3978)
centroids["geometry"] = centroids.geometry.centroid

# 3. Reproject centroids to WGS84 for web plotting
centroids_wgs = centroids.to_crs(4326)

# 4. Extract true lon/lat in degrees
centroids_wgs["lon"] = centroids_wgs.geometry.x
centroids_wgs["lat"] = centroids_wgs.geometry.y

In [ ]:
# For animation, we only need the fire points df
fig_fires = px.scatter_map(
    centroids_wgs,
    lat="lat",
    lon="lon",
    color="subreg_ha",
    animation_frame="year",
    map_style="carto-positron",
    center={"lat": 54, "lon": -123},
    zoom=5,
    labels={
        "gid":"Fire ID",
        "subregion": "Subregion name",
        "prov_terr": "Province/Territory",
        "n_fires": "Number of fires",
        'subreg_ha':'Hectares Burnt'
    },
    hover_name="gid",
    hover_data={
        "gid": True,
        "prov_terr": True,
        "region": True,
        "subregion": True,
        "subreg_ha": ':.0f',
    }
)

# Add AvCan polygons as a GeoJSON layer under the points
fig_fires.update_layout(
    map=dict(
        style="carto-positron",
        center={"lat": 54, "lon": -123},
        zoom=5,
        layers=[
            dict(
                source=avcan_geojson,
                type="fill",
                below="traces",
                color="rgba(0,0,0,0.05)",   # very light tint so it doesn’t overpower fires
            )
        ],
    ),
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
)

fig_fires.show()


## Fire perimeters

In [ ]:
fire_stats['region'].unique()

In [ ]:
# # Example fire id from your screenshot
# gids = fire_stats[fire_stats['subregion']=='Brandywine']['gid']

# Regions you care about

# region_sel = "Sea To Sky"

# subregions = (
#     fire_stats.loc[fire_stats["region"] == region_sel, "subregion"]
#     .dropna()
#     .unique()
#     .tolist()
# )

subregions = ['Birkenhead']



# AvCan subregions of interest
subregs = avcan_regions[avcan_regions["subregion"].isin(subregions)]

# Fires inside those subregions
fires_sel = fire_stats[fire_stats["subregion"].isin(subregions)]



In [ ]:
subregions


In [ ]:
subregion_str = ", ".join(subregions) 

subregion_str

2. Reproject to WGS84 for Plotly

Plotly wants lon/lat in EPSG:4326.

In [ ]:
fires_sel_wgs   = fires_sel.to_crs(4326)
subregs_wgs = subregs.to_crs(4326)


In [ ]:

    

subregion_fires_path_geojson = processed_dir / "subregion_fires" / f"{subregion_str}_fires_shp_files"
print(f'Exporting {str(subregion_str)} fires as GeoJSON...')

try:
    fires_sel_wgs.to_file(subregion_fires_path_geojson, driver="ESRI Shapefile")
    print(f'{subregion_str} fires GeoJSON export successful: {subregion_fires_path_geojson}')
except Exception as e:
    raise RuntimeError(f'{subregion_str} fires GeoJSON failed to export: {e}')
